In [1]:
import numpy as np 
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

In [2]:
df = pd.read_excel("Reestr.xlsx", nrows=5000, header = 2)

c:\Users\s7omb\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
df.columns

Index(['№ п/п', 'Наименование / ФИО', 'Тип субъекта', 'Категория', 'ОГРН',
       'ИНН', 'Основной вид деятельности', 'Регион', 'Район', 'Город',
       'Населенный пункт', 'Вновь созданный', 'Дата включения в реестр',
       'Дата исключения из реестра', 'Телефон', 'E-mail', 'WWW',
       'Наличие лицензий', 'Наличие заключенных договоров, контрактов',
       'Производство инновационной, высокотехнологичной продукции',
       'Участие в программах партнерства', 'Является социальным предприятием',
       'Среднесписочная численность работников за предшествующий календарный год'],
      dtype='object')

In [4]:
need_columns = ["№ п/п", 'Наименование / ФИО', 'Тип субъекта', "Категория", "ИНН", "Основной вид деятельности"]

In [5]:
df = df[need_columns]
df_filter = df[(df["Тип субъекта"] == "Юридическое лицо") & ((df["Категория"] == "Малое предприятие") | (df["Категория"] == "Среднее предприятие")) & (df["Основной вид деятельности"].str.startswith("41.2", na=False)) ]

In [6]:
inn = df_filter["ИНН"].to_numpy()

In [46]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept": "*/*",
    "Accept-Language": "ru,en;q=0.9",
}

code_company = []

for i in inn[:15]:   
    session = requests.Session()
    
    res = session.get(f"https://bo.nalog.gov.ru/advanced-search/organizations/search?query={i}", headers=headers, timeout=20).json()
    time.sleep(1)
    if res["content"]:
        code_company.append(res["content"][0]["id"])

code_company 

[5872486,
 5972153,
 6900709,
 2996748,
 4980670,
 3867230,
 7041303,
 6840082,
 5446585,
 7244164,
 2608351,
 4659067,
 2083796,
 5591507]

In [49]:

ebit_data = {} 

for i, code in zip(inn, code_company[:15]):
        ebit_data[int(i)] = {}

        try: 
                bfo_data = requests.get(f"https://bo.nalog.gov.ru/nbo/organizations/{code}/bfo/", headers=headers, timeout=20).json()
                

        except Exception as e:
                print(f"Ошибка запроса для ИНН {inn}: {e}")
                continue



        for period in bfo_data:

                year = int(period["period"])

                financial = period["typeCorrections"][0]["correction"]["financialResult"]

                profit_before_tax = financial.get("current2300", 0)
                interest_expense = financial.get("current2330", 0)
                final_ebit = profit_before_tax + interest_expense 

                ebit_data[i][year]= final_ebit

ebit_data

{5038038838: {2023: -17331.0,
  2021: 9031.0,
  2024: -18417.0,
  2022: 20360.0,
  2025: 11412.0},
 5027064258: {2021: 4721.0,
  2025: 6217.0,
  2023: 6593.0,
  2024: 6295.0,
  2022: 3068.0},
 5027006369: {2022: 42801.0,
  2025: 753572.0,
  2024: 37887.0,
  2021: 25839.0,
  2023: 730016.0},
 7701651356: {2025: 24862.0,
  2024: 210653.0,
  2021: 20108.0,
  2022: 3262.0,
  2023: -567.0},
 1414006922: {2021: 7940.0,
  2025: 83370.0,
  2024: 71787.0,
  2023: 28513.0,
  2022: 7409.0},
 3327332190: {2024: 61184.0,
  2023: 50126.0,
  2022: 38067.0,
  2021: 26125.0,
  2025: 88371.0},
 7816061297: {2023: 0.0, 2022: 574.0, 2021: 1385.0, 2025: 0, 2024: 0},
 3525048992: {2021: 16672.0,
  2023: 243530.0,
  2025: 195980.0,
  2024: 104109.0,
  2022: 14755.0},
 2515000365: {2021: 4091.0,
  2025: 1340.0,
  2023: 8328.0,
  2024: 7448.0,
  2022: 45038.0},
 3101000612: {2021: -3036.0,
  2024: -20819.0,
  2025: -19226.0,
  2023: -13946.0,
  2022: -12667.0},
 5902126804: {2021: 11054.0,
  2024: 3017.0,
  20

In [ ]:
financial = bfo_data[0]["typeCorrections"][0]["correction"]["financialResult"]

profit_before_tax = financial.get("current2300", 0)
interest_expense = financial.get("current2330", 0)

profit_before_tax

40010.0

In [ ]:
for period_data in bfo_data:
    print(period_data.get("period"))

2022
2025
2024
2021
2023
